# 502 — Integrated Vulnerability Mapping

## Objective

Integrate the frozen CRISPR and RNAi functional-genomics association layers from
notebooks 500 and 501 to characterize cross-platform correspondence between the
three frozen Phase 4 consensus transcriptomic programs and gene-level dependency
signals.

This notebook performs evidence integration, not statistical pooling. CRISPR and
RNAi remain distinct measurement platforms with separate dependency scales,
coverage properties, model composition, technical structure, and uncertainty.

The objective is therefore to construct a transparent program × gene evidence map
that preserves platform-specific results while describing their correspondence.

## Frozen upstream inputs

Notebook 502 consumes only frozen outputs from the completed platform-specific
analyses:

- notebook 500 CRISPR association outputs;
- notebook 501 RNAi association outputs;
- the frozen Phase 4 consensus-program identities:
  `CONSENSUS_TX_01`, `CONSENSUS_TX_02`, and `CONSENSUS_TX_03`;
- platform-specific gene identifiers, eligibility annotations, effect estimates,
  FDR values, coverage information, and lineage-aware characterization already
  produced upstream.

Notebook 502 will not refit, reorient, reweight, rescue, exclude, or otherwise
redefine CRISPR, RNAi, or consensus-program results.

## Analytical unit

The primary integration unit is:

`frozen consensus program × gene`

Cross-platform correspondence is established only for gene-level analytical units
that can be mapped deterministically between the frozen CRISPR and RNAi outputs.

Exact identifiers are preferred for correspondence. Gene symbols may be retained
for interpretation but will not be used to resolve ambiguous mappings
post hoc.

Genes that cannot be mapped unambiguously across platforms remain valid
platform-specific observations and are explicitly classified as not comparable
rather than discarded or reassigned.

## Platform-specific eligibility boundary

Cross-platform comparability preserves the primary eligibility criteria already
frozen within each platform-specific analysis.

The CRISPR primary universe from notebook 500 uses its frozen >=90% model-coverage
eligibility criterion, whereas the RNAi primary universe from notebook 501 uses
its frozen >=75% model-coverage criterion for single-gene DEMETER2 targets.

Notebook 502 does not impose a new common coverage threshold across platforms.
A program × gene hypothesis is therefore classified as cross-platform comparable
when that gene is eligible under the frozen primary criterion of each platform,
not under a retrospectively harmonized coverage rule.

The >=90% RNAi coverage analysis remains a prespecified sensitivity analysis from
notebook 501. It is retained only as platform-specific contextual evidence and
does not redefine RNAi primary eligibility or cross-platform comparability.

## Cross-platform evidence categories

Before inspection of joint CRISPR–RNAi results, the following correspondence
categories are prespecified:

- **concordant putative-vulnerability association**:
  CRISPR and RNAi are both FDR-significant for the same program × gene pair and
  both effect directions are compatible with stronger dependency at higher
  consensus-program score;

- **concordant weaker-dependency association**:
  CRISPR and RNAi are both FDR-significant and both effect directions are
  compatible with weaker dependency at higher consensus-program score;

- **CRISPR-supported only**:
  the association is FDR-significant in CRISPR but not in RNAi among
  cross-platform-comparable hypotheses;

- **RNAi-supported only**:
  the association is FDR-significant in RNAi but not in CRISPR among
  cross-platform-comparable hypotheses;

- **cross-platform directional discordance**:
  both platforms provide FDR-significant evidence for the same program × gene
  pair but their dependency-effect directions differ;

- **comparable, non-significant in both**:
  the program × gene pair is evaluable in both platforms but neither platform
  meets its frozen FDR criterion;

- **not cross-platform comparable**:
  the gene is absent, ineligible, composite, ambiguously mapped, or otherwise
  unavailable as a directly comparable analytical unit in at least one platform.

These categories describe correspondence between frozen evidence layers. They do
not redefine significance within either platform.

## Integration principles

CRISPR and RNAi evidence will be retained as separate evidence dimensions.

Notebook 502 will not:

- average or otherwise pool CRISPR and RNAi dependency scores;
- combine CRISPR and RNAi regression coefficients;
- combine platform-specific p-values or q-values;
- recalculate a joint cross-platform FDR from platform-specific significance
  results;
- construct a composite vulnerability score or ranking;
- use cross-platform concordance as a retrospective eligibility or significance
  gate;
- remove platform-specific associations because the other platform does not
  reproduce them;
- promote discordant or single-platform signals through post-hoc biological
  preference;
- redefine upstream lineage, coverage, source, or sensitivity criteria.

## Interpretation of concordance

Cross-platform concordance may strengthen the descriptive case that a candidate
program–dependency association is reproducible across distinct functional-genomics
assays.

However, concordance is not treated as independent experimental validation.
CRISPR and RNAi can share cell-line context, biological confounders, gene-level
dependency structure, and partially overlapping model populations.

Accordingly, cross-platform agreement is interpreted as complementary
computational evidence rather than causal confirmation.

Likewise, cross-platform discordance is not considered an analytical failure.
Discordance may reflect platform-specific biology, assay characteristics,
coverage differences, model composition, lineage structure, measurement noise,
or other unresolved technical and biological factors.

## Lineage-aware boundary

Lineage-aware estimates and heterogeneity characterizations produced in notebooks
500 and 501 remain platform-specific evidence dimensions.

Notebook 502 may compare their directional patterns descriptively but will not
introduce a new lineage-consistency threshold, heterogeneity gate, or pan-cancer
uniformity requirement.

Program–gene associations may therefore remain biologically context-dependent
even when cross-platform correspondence is observed.

## Scope boundary

Notebook 502 is restricted to integrated functional-vulnerability mapping.

It will not perform:

- pharmacogenomic association testing;
- drug prioritization;
- LINCS or CMap perturbational analysis;
- compound ranking;
- therapeutic-response prediction;
- causal inference;
- clinical prediction;
- target validation.

Downstream pharmacogenomic and perturbational analyses, if performed, must consume
the frozen outputs of this notebook without retrospectively redefining the
CRISPR–RNAi evidence categories.

The final output of notebook 502 is therefore a transparent, platform-aware
cross-platform evidence map rather than a combined predictor or definitive
vulnerability ranking.

In [1]:
# =============================================================================
# Imports
# =============================================================================

import json

import numpy as np
import pandas as pd

from pancancer_epigenetics.utils.paths import (
    Paths,
    project_relative_path,
)

In [2]:
# =============================================================================
# Input paths
# =============================================================================

CRISPR_MODEL_COHORT_PATH = (
    Paths.dependencies
    / "500_crispr_model_cohort.csv"
)

CRISPR_GENE_COVERAGE_PATH = (
    Paths.dependencies
    / "500_crispr_gene_coverage.csv"
)

CRISPR_PRIMARY_ASSOCIATIONS_PATH = (
    Paths.functional_vulnerabilities
    / "500_crispr_primary_associations.csv"
)

CRISPR_WITHIN_LINEAGE_PATH = (
    Paths.functional_vulnerabilities
    / "500_crispr_within_lineage_associations.csv"
)

RNAI_MODEL_COHORT_PATH = (
    Paths.dependencies
    / "501_rnai_model_cohort.csv"
)

RNAI_GENE_ELIGIBILITY_PATH = (
    Paths.dependencies
    / "501_rnai_gene_eligibility.csv"
)

RNAI_PRIMARY_ASSOCIATIONS_PATH = (
    Paths.functional_vulnerabilities
    / "501_rnai_primary_associations.csv"
)

RNAI_WITHIN_LINEAGE_PATH = (
    Paths.functional_vulnerabilities
    / "501_rnai_within_lineage_associations.csv"
)

RNAI_SOURCE_ADJUSTED_PATH = (
    Paths.functional_vulnerabilities
    / "501_rnai_source_adjusted_sensitivity.csv"
)

RNAI_COVERAGE90_PATH = (
    Paths.functional_vulnerabilities
    / "501_rnai_coverage90_sensitivity.csv"
)

In [3]:
# =============================================================================
# Load frozen functional-genomics inputs
# =============================================================================

crispr_model_cohort = pd.read_csv(
    CRISPR_MODEL_COHORT_PATH
)

crispr_gene_coverage = pd.read_csv(
    CRISPR_GENE_COVERAGE_PATH
)

crispr_primary_associations = pd.read_csv(
    CRISPR_PRIMARY_ASSOCIATIONS_PATH
)

crispr_within_lineage = pd.read_csv(
    CRISPR_WITHIN_LINEAGE_PATH
)

rnai_model_cohort = pd.read_csv(
    RNAI_MODEL_COHORT_PATH
)

rnai_gene_eligibility = pd.read_csv(
    RNAI_GENE_ELIGIBILITY_PATH
)

rnai_primary_associations = pd.read_csv(
    RNAI_PRIMARY_ASSOCIATIONS_PATH
)

rnai_within_lineage = pd.read_csv(
    RNAI_WITHIN_LINEAGE_PATH
)

rnai_source_adjusted = pd.read_csv(
    RNAI_SOURCE_ADJUSTED_PATH
)

rnai_coverage90 = pd.read_csv(
    RNAI_COVERAGE90_PATH
)

In [4]:
# =============================================================================
# Prespecified integration parameters
# =============================================================================

FROZEN_CONSENSUS_PROGRAM_IDS = [
    "CONSENSUS_TX_01",
    "CONSENSUS_TX_02",
    "CONSENSUS_TX_03",
]

STRONGER_DEPENDENCY_DIRECTION = (
    "higher_program_stronger_dependency"
)

WEAKER_DEPENDENCY_DIRECTION = (
    "higher_program_weaker_dependency"
)

CROSS_PLATFORM_EVIDENCE_CATEGORIES = [
    "concordant_putative_vulnerability",
    "concordant_weaker_dependency",
    "crispr_supported_only",
    "rnai_supported_only",
    "directionally_discordant",
    "non_significant_both",
    "not_cross_platform_comparable",
]

In [5]:
# =============================================================================
# Characterize CRISPR–RNAi model overlap
# =============================================================================

crispr_model_ids = set(
    crispr_model_cohort["ModelID"]
)

rnai_model_ids = set(
    rnai_model_cohort["ModelID"]
)

shared_model_ids = (
    crispr_model_ids
    & rnai_model_ids
)

crispr_only_model_ids = (
    crispr_model_ids
    - rnai_model_ids
)

rnai_only_model_ids = (
    rnai_model_ids
    - crispr_model_ids
)

model_overlap_summary = pd.Series(
    {
        "crispr_models": len(crispr_model_ids),
        "rnai_models": len(rnai_model_ids),
        "shared_models": len(shared_model_ids),
        "crispr_only_models": len(crispr_only_model_ids),
        "rnai_only_models": len(rnai_only_model_ids),
        "shared_fraction_crispr": (
            len(shared_model_ids)
            / len(crispr_model_ids)
        ),
        "shared_fraction_rnai": (
            len(shared_model_ids)
            / len(rnai_model_ids)
        ),
    },
    name="value",
).to_frame()

model_overlap_summary

,value
crispr_models,539.000000
rnai_models,443.000000
shared_models,367.000000
crispr_only_models,172.000000
rnai_only_models,76.000000
shared_fraction_crispr,0.680891
shared_fraction_rnai,0.828442


In [6]:
# =============================================================================
# Characterize CRISPR–RNAi lineage overlap
# =============================================================================

crispr_lineages = set(
    crispr_model_cohort["OncotreeLineage"].dropna()
)

rnai_lineages = set(
    rnai_model_cohort["OncotreeLineage"].dropna()
)

shared_lineages = (
    crispr_lineages
    & rnai_lineages
)

crispr_only_lineages = (
    crispr_lineages
    - rnai_lineages
)

rnai_only_lineages = (
    rnai_lineages
    - crispr_lineages
)

lineage_overlap_summary = pd.Series(
    {
        "crispr_lineages": len(crispr_lineages),
        "rnai_lineages": len(rnai_lineages),
        "shared_lineages": len(shared_lineages),
        "crispr_only_lineages": len(crispr_only_lineages),
        "rnai_only_lineages": len(rnai_only_lineages),
    },
    name="value",
).to_frame()

lineage_overlap_summary

,value
crispr_lineages,26
rnai_lineages,22
shared_lineages,22
crispr_only_lineages,4
rnai_only_lineages,0


In [7]:
# =============================================================================
# Prepare cross-platform gene identifier tables
# =============================================================================

crispr_eligible_genes = (
    crispr_gene_coverage.loc[
        crispr_gene_coverage["analysis_eligible"],
        [
            "source_gene_label",
            "gene_symbol",
            "entrez_id",
        ],
    ]
    .rename(
        columns={
            "source_gene_label": "crispr_gene_label",
            "gene_symbol": "crispr_gene_symbol",
            "entrez_id": "cross_platform_entrez_id",
        }
    )
    .copy()
)

rnai_eligible_genes = (
    rnai_gene_eligibility.loc[
        rnai_gene_eligibility["primary_gene_eligible"],
        [
            "gene_label",
            "gene_symbol_group",
            "entrez_id_group",
        ],
    ]
    .rename(
        columns={
            "gene_label": "rnai_gene_label",
            "gene_symbol_group": "rnai_gene_symbol",
            "entrez_id_group": "cross_platform_entrez_id",
        }
    )
    .copy()
)

In [8]:
# =============================================================================
# Characterize cross-platform gene identifier integrity
# =============================================================================

gene_identifier_integrity = pd.DataFrame(
    {
        "platform": [
            "CRISPR",
            "RNAi",
        ],
        "eligible_genes": [
            len(crispr_eligible_genes),
            len(rnai_eligible_genes),
        ],
        "missing_entrez_id": [
            crispr_eligible_genes[
                "cross_platform_entrez_id"
            ].isna().sum(),
            rnai_eligible_genes[
                "cross_platform_entrez_id"
            ].isna().sum(),
        ],
        "duplicated_entrez_id_rows": [
            crispr_eligible_genes[
                "cross_platform_entrez_id"
            ].duplicated(keep=False).sum(),
            rnai_eligible_genes[
                "cross_platform_entrez_id"
            ].duplicated(keep=False).sum(),
        ],
        "unique_entrez_ids": [
            crispr_eligible_genes[
                "cross_platform_entrez_id"
            ].nunique(dropna=True),
            rnai_eligible_genes[
                "cross_platform_entrez_id"
            ].nunique(dropna=True),
        ],
    }
)

gene_identifier_integrity

,platform,eligible_genes,missing_entrez_id,duplicated_entrez_id_rows,unique_entrez_ids
0,CRISPR,17205,0,0,17205
1,RNAi,12598,0,0,12598


In [9]:
# =============================================================================
# Harmonize cross-platform Entrez identifier type
# =============================================================================

crispr_eligible_genes["cross_platform_entrez_id"] = (
    pd.to_numeric(
        crispr_eligible_genes["cross_platform_entrez_id"],
        errors="raise",
    )
    .astype("Int64")
)

rnai_eligible_genes["cross_platform_entrez_id"] = (
    pd.to_numeric(
        rnai_eligible_genes["cross_platform_entrez_id"],
        errors="raise",
    )
    .astype("Int64")
)

pd.Series(
    {
        "crispr_entrez_dtype": str(
            crispr_eligible_genes["cross_platform_entrez_id"].dtype
        ),
        "rnai_entrez_dtype": str(
            rnai_eligible_genes["cross_platform_entrez_id"].dtype
        ),
    },
    name="value",
).to_frame()

,value
crispr_entrez_dtype,Int64
rnai_entrez_dtype,Int64


In [10]:
# =============================================================================
# Construct cross-platform gene map
# =============================================================================

cross_platform_gene_map = (
    crispr_eligible_genes
    .merge(
        rnai_eligible_genes,
        on="cross_platform_entrez_id",
        how="outer",
        validate="one_to_one",
        indicator="platform_membership",
    )
    .copy()
)

In [11]:
# =============================================================================
# Characterize cross-platform gene overlap
# =============================================================================

gene_overlap_summary = pd.Series(
    {
        "crispr_eligible_genes": (
            cross_platform_gene_map["platform_membership"]
            .isin(["left_only", "both"])
            .sum()
        ),
        "rnai_eligible_genes": (
            cross_platform_gene_map["platform_membership"]
            .isin(["right_only", "both"])
            .sum()
        ),
        "shared_eligible_genes": (
            cross_platform_gene_map["platform_membership"]
            .eq("both")
            .sum()
        ),
        "crispr_only_genes": (
            cross_platform_gene_map["platform_membership"]
            .eq("left_only")
            .sum()
        ),
        "rnai_only_genes": (
            cross_platform_gene_map["platform_membership"]
            .eq("right_only")
            .sum()
        ),
    },
    name="value",
).to_frame()

gene_overlap_summary

,value
crispr_eligible_genes,17205
rnai_eligible_genes,12598
shared_eligible_genes,11486
crispr_only_genes,5719
rnai_only_genes,1112


In [12]:
# =============================================================================
# Characterize shared-gene symbol concordance
# =============================================================================

shared_gene_map = (
    cross_platform_gene_map.loc[
        cross_platform_gene_map["platform_membership"].eq("both")
    ]
    .copy()
)

shared_gene_map["gene_symbol_exact_match"] = (
    shared_gene_map["crispr_gene_symbol"]
    .eq(shared_gene_map["rnai_gene_symbol"])
)

gene_symbol_concordance_summary = pd.Series(
    {
        "shared_genes": len(shared_gene_map),
        "missing_crispr_symbol": (
            shared_gene_map["crispr_gene_symbol"].isna().sum()
        ),
        "missing_rnai_symbol": (
            shared_gene_map["rnai_gene_symbol"].isna().sum()
        ),
        "exact_symbol_matches": (
            shared_gene_map["gene_symbol_exact_match"].sum()
        ),
        "symbol_mismatches": (
            (~shared_gene_map["gene_symbol_exact_match"]).sum()
        ),
    },
    name="value",
).to_frame()

gene_symbol_concordance_summary

,value
shared_genes,11486
missing_crispr_symbol,0
missing_rnai_symbol,0
exact_symbol_matches,11279
symbol_mismatches,207


In [13]:
# =============================================================================
# Inspect shared-gene symbol mismatches
# =============================================================================

gene_symbol_mismatches = (
    shared_gene_map.loc[
        ~shared_gene_map["gene_symbol_exact_match"],
        [
            "cross_platform_entrez_id",
            "crispr_gene_label",
            "crispr_gene_symbol",
            "rnai_gene_label",
            "rnai_gene_symbol",
        ],
    ]
    .sort_values("cross_platform_entrez_id")
    .reset_index(drop=True)
)

gene_symbol_mismatches

,cross_platform_entrez_id,crispr_gene_label,crispr_gene_symbol,rnai_gene_label,rnai_gene_symbol
0,16,AARS1 (16),AARS1,AARS (16),AARS
1,55,ACP3 (55),ACP3,ACPP (55),ACPP
2,159,ADSS2 (159),ADSS2,ADSS (159),ADSS
3,166,TLE5 (166),TLE5,AES (166),AES
4,251,ALPG (251),ALPG,ALPPL2 (251),ALPPL2
...,...,...,...,...,...
202,339834,IHO1 (339834),IHO1,CCDC36 (339834),CCDC36
203,341567,H1-7 (341567),H1-7,H1FNT (341567),H1FNT
204,387521,PEDS1 (387521),PEDS1,TMEM189 (387521),TMEM189
205,388650,DIPK1A (388650),DIPK1A,FAM69A (388650),FAM69A


In [14]:
# =============================================================================
# Define cross-platform gene comparability
# =============================================================================

cross_platform_gene_map["cross_platform_comparable"] = (
    cross_platform_gene_map["platform_membership"].eq("both")
)

cross_platform_gene_map["comparability_status"] = (
    cross_platform_gene_map["platform_membership"]
    .map(
        {
            "both": "cross_platform_comparable",
            "left_only": "crispr_only",
            "right_only": "rnai_only",
        }
    )
    .astype("string")
)

cross_platform_gene_map["gene_symbol_exact_match"] = (
    cross_platform_gene_map["crispr_gene_symbol"]
    .eq(cross_platform_gene_map["rnai_gene_symbol"])
    .where(
        cross_platform_gene_map["cross_platform_comparable"]
    )
)

In [15]:
# =============================================================================
# Inspect primary-association integration fields
# =============================================================================

integration_field_inventory = pd.DataFrame(
    {
        "crispr_columns": pd.Series(
            crispr_primary_associations.columns,
            dtype="string",
        ),
        "rnai_columns": pd.Series(
            rnai_primary_associations.columns,
            dtype="string",
        ),
    }
)

integration_field_inventory

,crispr_columns,rnai_columns
0,consensus_program_id,consensus_program_id
1,source_gene_label,gene_label
2,gene_symbol,n_models
3,entrez_id,n_lineages
4,n_models,beta
5,beta,standard_error
6,standard_error,ci_95_lower
7,p_value,ci_95_upper
8,fdr_q_value,p_value
9,fdr_significant,fdr_q_value


In [16]:
# =============================================================================
# Inspect frozen effect-direction representation
# =============================================================================

effect_direction_inventory = pd.Series(
    {
        "rnai_effect_directions": tuple(
            sorted(
                rnai_primary_associations[
                    "effect_direction"
                ]
                .dropna()
                .unique()
            )
        ),
        "crispr_beta_min": (
            crispr_primary_associations["beta"].min()
        ),
        "crispr_beta_max": (
            crispr_primary_associations["beta"].max()
        ),
        "rnai_beta_min": (
            rnai_primary_associations["beta"].min()
        ),
        "rnai_beta_max": (
            rnai_primary_associations["beta"].max()
        ),
    },
    name="value",
).to_frame()

effect_direction_inventory

,value
rnai_effect_directions,"(higher_program_stronger_dependency, higher_pr..."
crispr_beta_min,-0.335312
crispr_beta_max,0.43901
rnai_beta_min,-0.351812
rnai_beta_max,0.310518


In [17]:
# =============================================================================
# Verify dependency-effect direction convention
# =============================================================================

rnai_direction_from_beta = np.select(
    [
        rnai_primary_associations["beta"].lt(0),
        rnai_primary_associations["beta"].gt(0),
    ],
    [
        STRONGER_DEPENDENCY_DIRECTION,
        WEAKER_DEPENDENCY_DIRECTION,
    ],
    default="zero_effect",
)

direction_convention_check = pd.crosstab(
    pd.Series(
        rnai_direction_from_beta,
        name="direction_from_beta",
    ),
    rnai_primary_associations[
        "effect_direction"
    ].rename("frozen_rnai_direction"),
    dropna=False,
)

direction_convention_check

frozen_rnai_direction,higher_program_stronger_dependency,higher_program_weaker_dependency
direction_from_beta,,
higher_program_stronger_dependency,18443,0
higher_program_weaker_dependency,0,19351


In [18]:
# =============================================================================
# Prepare primary associations for cross-platform integration
# =============================================================================

crispr_primary_for_integration = (
    crispr_primary_associations[
        [
            "consensus_program_id",
            "source_gene_label",
            "gene_symbol",
            "entrez_id",
            "n_models",
            "beta",
            "standard_error",
            "p_value",
            "fdr_q_value",
            "fdr_significant",
        ]
    ]
    .rename(
        columns={
            "source_gene_label": "crispr_gene_label",
            "gene_symbol": "crispr_gene_symbol",
            "entrez_id": "cross_platform_entrez_id",
            "n_models": "crispr_n_models",
            "beta": "crispr_beta",
            "standard_error": "crispr_standard_error",
            "p_value": "crispr_p_value",
            "fdr_q_value": "crispr_fdr_q_value",
            "fdr_significant": "crispr_fdr_significant",
        }
    )
    .copy()
)

crispr_primary_for_integration["crispr_effect_direction"] = np.select(
    [
        crispr_primary_for_integration["crispr_beta"].lt(0),
        crispr_primary_for_integration["crispr_beta"].gt(0),
    ],
    [
        STRONGER_DEPENDENCY_DIRECTION,
        WEAKER_DEPENDENCY_DIRECTION,
    ],
    default="zero_effect",
)

rnai_primary_for_integration = (
    rnai_primary_associations[
        [
            "consensus_program_id",
            "gene_label",
            "gene_symbol",
            "entrez_id",
            "n_models",
            "n_lineages",
            "beta",
            "standard_error",
            "ci_95_lower",
            "ci_95_upper",
            "p_value",
            "fdr_q_value",
            "fdr_significant",
            "effect_direction",
            "rnai_observed_models",
            "rnai_coverage_fraction",
        ]
    ]
    .rename(
        columns={
            "gene_label": "rnai_gene_label",
            "gene_symbol": "rnai_gene_symbol",
            "entrez_id": "cross_platform_entrez_id",
            "n_models": "rnai_n_models",
            "n_lineages": "rnai_n_lineages",
            "beta": "rnai_beta",
            "standard_error": "rnai_standard_error",
            "ci_95_lower": "rnai_ci_95_lower",
            "ci_95_upper": "rnai_ci_95_upper",
            "p_value": "rnai_p_value",
            "fdr_q_value": "rnai_fdr_q_value",
            "fdr_significant": "rnai_fdr_significant",
            "effect_direction": "rnai_effect_direction",
        }
    )
    .copy()
)

In [19]:
# =============================================================================
# Harmonize primary-association Entrez identifier type
# =============================================================================

crispr_primary_for_integration["cross_platform_entrez_id"] = (
    pd.to_numeric(
        crispr_primary_for_integration["cross_platform_entrez_id"],
        errors="raise",
    )
    .astype("Int64")
)

rnai_primary_for_integration["cross_platform_entrez_id"] = (
    pd.to_numeric(
        rnai_primary_for_integration["cross_platform_entrez_id"],
        errors="raise",
    )
    .astype("Int64")
)

In [20]:
# =============================================================================
# Verify program × gene integration-key uniqueness
# =============================================================================

integration_key_summary = pd.DataFrame(
    {
        "platform": ["CRISPR", "RNAi"],
        "rows": [
            len(crispr_primary_for_integration),
            len(rnai_primary_for_integration),
        ],
        "missing_key_rows": [
            crispr_primary_for_integration[
                ["consensus_program_id", "cross_platform_entrez_id"]
            ].isna().any(axis=1).sum(),
            rnai_primary_for_integration[
                ["consensus_program_id", "cross_platform_entrez_id"]
            ].isna().any(axis=1).sum(),
        ],
        "duplicated_key_rows": [
            crispr_primary_for_integration.duplicated(
                ["consensus_program_id", "cross_platform_entrez_id"],
                keep=False,
            ).sum(),
            rnai_primary_for_integration.duplicated(
                ["consensus_program_id", "cross_platform_entrez_id"],
                keep=False,
            ).sum(),
        ],
    }
)

integration_key_summary

,platform,rows,missing_key_rows,duplicated_key_rows
0,CRISPR,51615,0,0
1,RNAi,37794,0,0


In [21]:
# =============================================================================
# Construct cross-platform primary association map
# =============================================================================

integrated_primary_map = (
    crispr_primary_for_integration
    .merge(
        rnai_primary_for_integration,
        on=[
            "consensus_program_id",
            "cross_platform_entrez_id",
        ],
        how="outer",
        validate="one_to_one",
        indicator="hypothesis_membership",
    )
    .copy()
)

In [22]:
# =============================================================================
# Characterize integrated hypothesis universe
# =============================================================================

integrated_hypothesis_summary = pd.Series(
    {
        "integrated_hypotheses": len(integrated_primary_map),
        "cross_platform_comparable": (
            integrated_primary_map["hypothesis_membership"]
            .eq("both")
            .sum()
        ),
        "crispr_only": (
            integrated_primary_map["hypothesis_membership"]
            .eq("left_only")
            .sum()
        ),
        "rnai_only": (
            integrated_primary_map["hypothesis_membership"]
            .eq("right_only")
            .sum()
        ),
        "expected_shared_hypotheses": (
            gene_overlap_summary.loc[
                "shared_eligible_genes",
                "value",
            ]
            * len(FROZEN_CONSENSUS_PROGRAM_IDS)
        ),
        "expected_crispr_only_hypotheses": (
            gene_overlap_summary.loc[
                "crispr_only_genes",
                "value",
            ]
            * len(FROZEN_CONSENSUS_PROGRAM_IDS)
        ),
        "expected_rnai_only_hypotheses": (
            gene_overlap_summary.loc[
                "rnai_only_genes",
                "value",
            ]
            * len(FROZEN_CONSENSUS_PROGRAM_IDS)
        ),
    },
    name="value",
).to_frame()

integrated_hypothesis_summary

,value
integrated_hypotheses,54951
cross_platform_comparable,34458
crispr_only,17157
rnai_only,3336
expected_shared_hypotheses,34458
expected_crispr_only_hypotheses,17157
expected_rnai_only_hypotheses,3336


In [23]:
# =============================================================================
# Classify cross-platform primary evidence
# =============================================================================

integrated_primary_map["comparability_status"] = (
    integrated_primary_map["hypothesis_membership"]
    .map(
        {
            "both": "cross_platform_comparable",
            "left_only": "crispr_only",
            "right_only": "rnai_only",
        }
    )
    .astype("string")
)

crispr_significant = (
    integrated_primary_map["crispr_fdr_significant"].eq(True)
)

rnai_significant = (
    integrated_primary_map["rnai_fdr_significant"].eq(True)
)

same_direction = (
    integrated_primary_map["crispr_effect_direction"]
    .eq(integrated_primary_map["rnai_effect_direction"])
)

both_comparable = (
    integrated_primary_map["comparability_status"]
    .eq("cross_platform_comparable")
)

integrated_primary_map["cross_platform_evidence_category"] = np.select(
    [
        (
            both_comparable
            & crispr_significant
            & rnai_significant
            & same_direction
            & integrated_primary_map["crispr_effect_direction"].eq(
                STRONGER_DEPENDENCY_DIRECTION
            )
        ),
        (
            both_comparable
            & crispr_significant
            & rnai_significant
            & same_direction
            & integrated_primary_map["crispr_effect_direction"].eq(
                WEAKER_DEPENDENCY_DIRECTION
            )
        ),
        (
            both_comparable
            & crispr_significant
            & rnai_significant
            & ~same_direction
        ),
        (
            both_comparable
            & crispr_significant
            & ~rnai_significant
        ),
        (
            both_comparable
            & ~crispr_significant
            & rnai_significant
        ),
        (
            both_comparable
            & ~crispr_significant
            & ~rnai_significant
        ),
    ],
    [
        "concordant_putative_vulnerability",
        "concordant_weaker_dependency",
        "directionally_discordant",
        "crispr_supported_only",
        "rnai_supported_only",
        "non_significant_both",
    ],
    default="not_cross_platform_comparable",
)

In [24]:
# =============================================================================
# Annotate cross-platform directional correspondence
# =============================================================================

integrated_primary_map["cross_platform_direction_concordant"] = (
    integrated_primary_map["crispr_effect_direction"]
    .eq(integrated_primary_map["rnai_effect_direction"])
    .where(both_comparable)
    .astype("boolean")
)

integrated_primary_map["both_platforms_fdr_significant"] = (
    crispr_significant
    & rnai_significant
).where(
    both_comparable
).astype("boolean")

In [25]:
# =============================================================================
# Summarize cross-platform evidence categories
# =============================================================================

cross_platform_evidence_summary = (
    integrated_primary_map
    .groupby(
        [
            "consensus_program_id",
            "cross_platform_evidence_category",
        ],
        observed=True,
    )
    .size()
    .rename("n_hypotheses")
    .reset_index()
)

cross_platform_evidence_summary

,consensus_program_id,cross_platform_evidence_category,n_hypotheses
0,CONSENSUS_TX_01,concordant_putative_vulnerability,16
1,CONSENSUS_TX_01,concordant_weaker_dependency,12
2,CONSENSUS_TX_01,crispr_supported_only,164
3,CONSENSUS_TX_01,directionally_discordant,4
4,CONSENSUS_TX_01,non_significant_both,10873
5,CONSENSUS_TX_01,not_cross_platform_comparable,6831
6,CONSENSUS_TX_01,rnai_supported_only,417
7,CONSENSUS_TX_02,concordant_putative_vulnerability,6
8,CONSENSUS_TX_02,concordant_weaker_dependency,29
9,CONSENSUS_TX_02,crispr_supported_only,155


In [26]:
# =============================================================================
# Summarize global cross-platform evidence
# =============================================================================

cross_platform_evidence_global_summary = (
    integrated_primary_map[
        "cross_platform_evidence_category"
    ]
    .value_counts()
    .reindex(
        CROSS_PLATFORM_EVIDENCE_CATEGORIES,
        fill_value=0,
    )
    .rename_axis(
        "cross_platform_evidence_category"
    )
    .rename(
        "n_hypotheses"
    )
    .reset_index()
)

cross_platform_evidence_global_summary

,cross_platform_evidence_category,n_hypotheses
0,concordant_putative_vulnerability,35
1,concordant_weaker_dependency,77
2,crispr_supported_only,532
3,rnai_supported_only,1452
4,directionally_discordant,6
5,non_significant_both,32356
6,not_cross_platform_comparable,20493


In [27]:
# =============================================================================
# Characterize directional correspondence among comparable hypotheses
# =============================================================================

comparable_primary_map = (
    integrated_primary_map.loc[
        both_comparable
    ]
    .copy()
)

both_significant_mask = (
    comparable_primary_map["both_platforms_fdr_significant"]
    .fillna(False)
)

directional_correspondence_summary = pd.Series(
    {
        "comparable_hypotheses": len(comparable_primary_map),
        "direction_concordant": (
            comparable_primary_map[
                "cross_platform_direction_concordant"
            ].sum()
        ),
        "direction_discordant": (
            (~comparable_primary_map[
                "cross_platform_direction_concordant"
            ]).sum()
        ),
        "both_platforms_fdr_significant": (
            both_significant_mask.sum()
        ),
        "both_significant_direction_concordant": (
            comparable_primary_map.loc[
                both_significant_mask,
                "cross_platform_direction_concordant",
            ].sum()
        ),
        "both_significant_direction_discordant": (
            (
                ~comparable_primary_map.loc[
                    both_significant_mask,
                    "cross_platform_direction_concordant",
                ]
            ).sum()
        ),
        "both_significant_concordance_fraction": (
            comparable_primary_map.loc[
                both_significant_mask,
                "cross_platform_direction_concordant",
            ].mean()
        ),
    },
    name="value",
).to_frame()

directional_correspondence_summary

,value
comparable_hypotheses,34458.000000
direction_concordant,18137.000000
direction_discordant,16321.000000
both_platforms_fdr_significant,118.000000
both_significant_direction_concordant,112.000000
both_significant_direction_discordant,6.000000
both_significant_concordance_fraction,0.949153


In [28]:
# =============================================================================
# Characterize global cross-platform effect-size correspondence
# =============================================================================

global_effect_correspondence = pd.Series(
    {
        "n_comparable_hypotheses": len(
            comparable_primary_map
        ),
        "spearman_beta": (
            comparable_primary_map[
                "crispr_beta"
            ].corr(
                comparable_primary_map[
                    "rnai_beta"
                ],
                method="spearman",
            )
        ),
    },
    name="value",
).to_frame()

global_effect_correspondence

,value
n_comparable_hypotheses,34458.000000
spearman_beta,0.104793


In [29]:
# =============================================================================
# Characterize program-specific effect-size correspondence
# =============================================================================

program_effect_correspondence = (
    comparable_primary_map
    .groupby(
        "consensus_program_id",
        observed=True,
    )
    .apply(
        lambda df: pd.Series(
            {
                "n_comparable_hypotheses": len(df),
                "spearman_beta": df["crispr_beta"].corr(
                    df["rnai_beta"],
                    method="spearman",
                ),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)

program_effect_correspondence

,consensus_program_id,n_comparable_hypotheses,spearman_beta
0,CONSENSUS_TX_01,11486.0,0.097495
1,CONSENSUS_TX_02,11486.0,0.105062
2,CONSENSUS_TX_03,11486.0,0.140001


In [30]:
# =============================================================================
# Characterize program-specific concordance among bilateral significant signals
# =============================================================================

program_significant_concordance = (
    comparable_primary_map.loc[
        comparable_primary_map[
            "both_platforms_fdr_significant"
        ].fillna(False)
    ]
    .groupby(
        "consensus_program_id",
        observed=True,
    )
    .agg(
        both_significant=(
            "cross_platform_direction_concordant",
            "size",
        ),
        direction_concordant=(
            "cross_platform_direction_concordant",
            "sum",
        ),
    )
    .reset_index()
)

program_significant_concordance[
    "direction_discordant"
] = (
    program_significant_concordance[
        "both_significant"
    ]
    - program_significant_concordance[
        "direction_concordant"
    ]
)

program_significant_concordance[
    "concordance_fraction"
] = (
    program_significant_concordance[
        "direction_concordant"
    ]
    / program_significant_concordance[
        "both_significant"
    ]
)

program_significant_concordance

,consensus_program_id,both_significant,direction_concordant,direction_discordant,concordance_fraction
0,CONSENSUS_TX_01,32,28,4,0.875
1,CONSENSUS_TX_02,35,35,0,1.0
2,CONSENSUS_TX_03,51,49,2,0.960784


In [31]:
# =============================================================================
# Inspect cross-platform directionally discordant associations
# =============================================================================

directionally_discordant_associations = (
    integrated_primary_map.loc[
        integrated_primary_map[
            "cross_platform_evidence_category"
        ].eq("directionally_discordant"),
        [
            "consensus_program_id",
            "cross_platform_entrez_id",
            "crispr_gene_symbol",
            "rnai_gene_symbol",
            "crispr_beta",
            "crispr_fdr_q_value",
            "crispr_effect_direction",
            "rnai_beta",
            "rnai_fdr_q_value",
            "rnai_effect_direction",
        ],
    ]
    .sort_values(
        [
            "consensus_program_id",
            "cross_platform_entrez_id",
        ]
    )
    .reset_index(drop=True)
)

directionally_discordant_associations

,consensus_program_id,cross_platform_entrez_id,crispr_gene_symbol,rnai_gene_symbol,crispr_beta,crispr_fdr_q_value,crispr_effect_direction,rnai_beta,rnai_fdr_q_value,rnai_effect_direction
0,CONSENSUS_TX_01,5695,PSMB7,PSMB7,0.214398,0.000391,higher_program_weaker_dependency,-0.144959,0.014260,higher_program_stronger_dependency
1,CONSENSUS_TX_01,51106,TFB1M,TFB1M,-0.156515,0.011964,higher_program_stronger_dependency,0.098578,0.013216,higher_program_weaker_dependency
2,CONSENSUS_TX_01,55794,DDX28,DDX28,-0.138460,0.000563,higher_program_stronger_dependency,0.107256,0.023900,higher_program_weaker_dependency
3,CONSENSUS_TX_01,92856,IMP4,IMP4,-0.124240,0.027050,higher_program_stronger_dependency,0.183465,0.023397,higher_program_weaker_dependency
4,CONSENSUS_TX_03,3145,HMBS,HMBS,0.041257,0.033315,higher_program_weaker_dependency,-0.047963,0.005907,higher_program_stronger_dependency
5,CONSENSUS_TX_03,83988,NCALD,NCALD,0.028936,0.005562,higher_program_weaker_dependency,-0.044050,0.002224,higher_program_stronger_dependency


In [32]:
# =============================================================================
# Annotate gene-symbol concordance in integrated map
# =============================================================================

integrated_primary_map["gene_symbol_exact_match"] = (
    integrated_primary_map["crispr_gene_symbol"]
    .eq(integrated_primary_map["rnai_gene_symbol"])
    .where(
        integrated_primary_map["comparability_status"]
        .eq("cross_platform_comparable")
    )
    .astype("boolean")
)

In [33]:
# =============================================================================
# Extract concordant putative-vulnerability associations
# =============================================================================

concordant_putative_vulnerability_associations = (
    integrated_primary_map.loc[
        integrated_primary_map[
            "cross_platform_evidence_category"
        ].eq("concordant_putative_vulnerability"),
        [
            "consensus_program_id",
            "cross_platform_entrez_id",
            "crispr_gene_symbol",
            "rnai_gene_symbol",
            "gene_symbol_exact_match",
            "crispr_beta",
            "crispr_fdr_q_value",
            "rnai_beta",
            "rnai_fdr_q_value",
        ],
    ]
    .sort_values(
        [
            "consensus_program_id",
            "cross_platform_entrez_id",
        ]
    )
    .reset_index(drop=True)
)

concordant_putative_vulnerability_associations

,consensus_program_id,cross_platform_entrez_id,crispr_gene_symbol,rnai_gene_symbol,gene_symbol_exact_match,crispr_beta,crispr_fdr_q_value,rnai_beta,rnai_fdr_q_value
0,CONSENSUS_TX_01,47,ACLY,ACLY,True,-0.171337,9.425930e-03,-0.197957,0.017793
1,CONSENSUS_TX_01,1198,CLK3,CLK3,True,-0.073062,4.893508e-02,-0.107136,0.008382
2,CONSENSUS_TX_01,1327,COX4I1,COX4I1,True,-0.149752,1.732935e-03,-0.176613,0.016999
3,CONSENSUS_TX_01,1737,DLAT,DLAT,True,-0.069449,3.406470e-02,-0.132873,0.008907
4,CONSENSUS_TX_01,2146,EZH2,EZH2,True,-0.172542,3.907408e-04,-0.205462,0.007635
5,CONSENSUS_TX_01,2186,BPTF,BPTF,True,-0.138172,3.738588e-02,-0.180020,0.044283
6,CONSENSUS_TX_01,3052,HCCS,HCCS,True,-0.115496,2.460831e-02,-0.201538,0.045290
7,CONSENSUS_TX_01,4771,NF2,NF2,True,-0.178176,6.116176e-03,-0.094086,0.044510
8,CONSENSUS_TX_01,5079,PAX5,PAX5,True,-0.165959,8.654471e-03,-0.157884,0.001835
9,CONSENSUS_TX_01,6541,SLC7A1,SLC7A1,True,-0.174841,1.820204e-02,-0.138432,0.027230


In [34]:
# =============================================================================
# Characterize recurrence of concordant putative vulnerabilities
# =============================================================================

concordant_putative_gene_recurrence = (
    concordant_putative_vulnerability_associations
    .groupby(
        [
            "cross_platform_entrez_id",
            "crispr_gene_symbol",
        ],
        observed=True,
    )
    .agg(
        n_consensus_programs=(
            "consensus_program_id",
            "nunique",
        ),
        consensus_program_ids=(
            "consensus_program_id",
            lambda values: tuple(sorted(values.unique())),
        ),
    )
    .reset_index()
    .sort_values(
        [
            "n_consensus_programs",
            "cross_platform_entrez_id",
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

concordant_putative_gene_recurrence

,cross_platform_entrez_id,crispr_gene_symbol,n_consensus_programs,consensus_program_ids
0,5573,PRKAR1A,2,"(CONSENSUS_TX_02, CONSENSUS_TX_03)"
1,47,ACLY,1,"(CONSENSUS_TX_01,)"
2,998,CDC42,1,"(CONSENSUS_TX_03,)"
3,1198,CLK3,1,"(CONSENSUS_TX_01,)"
4,1327,COX4I1,1,"(CONSENSUS_TX_01,)"
5,1737,DLAT,1,"(CONSENSUS_TX_01,)"
6,1871,E2F3,1,"(CONSENSUS_TX_02,)"
7,2146,EZH2,1,"(CONSENSUS_TX_01,)"
8,2186,BPTF,1,"(CONSENSUS_TX_01,)"
9,2260,FGFR1,1,"(CONSENSUS_TX_03,)"


In [35]:
# =============================================================================
# Inspect lineage-characterization integration fields
# =============================================================================

lineage_field_inventory = pd.DataFrame(
    {
        "crispr_columns": pd.Series(
            crispr_within_lineage.columns,
            dtype="string",
        ),
        "rnai_columns": pd.Series(
            rnai_within_lineage.columns,
            dtype="string",
        ),
    }
)

lineage_field_inventory

,crispr_columns,rnai_columns
0,consensus_program_id,consensus_program_id
1,source_gene_label,gene_label
2,gene_symbol,OncotreeLineage
3,entrez_id,n_models
4,OncotreeLineage,beta
5,n_models,standard_error
6,beta,ci_95_lower
7,<NA>,ci_95_upper
8,<NA>,primary_beta
9,<NA>,direction_consistent


In [36]:
# =============================================================================
# Prepare lineage-aware associations for integration
# =============================================================================

crispr_lineage_for_integration = (
    crispr_within_lineage[
        [
            "consensus_program_id",
            "source_gene_label",
            "gene_symbol",
            "entrez_id",
            "OncotreeLineage",
            "n_models",
            "beta",
        ]
    ]
    .rename(
        columns={
            "source_gene_label": "crispr_gene_label",
            "gene_symbol": "crispr_gene_symbol",
            "entrez_id": "cross_platform_entrez_id",
            "OncotreeLineage": "crispr_lineage",
            "n_models": "crispr_lineage_n_models",
            "beta": "crispr_lineage_beta",
        }
    )
    .copy()
)

rnai_lineage_for_integration = (
    rnai_within_lineage[
        [
            "consensus_program_id",
            "gene_label",
            "gene_symbol",
            "entrez_id",
            "OncotreeLineage",
            "n_models",
            "beta",
            "primary_beta",
            "direction_consistent",
        ]
    ]
    .rename(
        columns={
            "gene_label": "rnai_gene_label",
            "gene_symbol": "rnai_gene_symbol",
            "entrez_id": "cross_platform_entrez_id",
            "OncotreeLineage": "rnai_lineage",
            "n_models": "rnai_lineage_n_models",
            "beta": "rnai_lineage_beta",
            "primary_beta": "rnai_primary_beta",
            "direction_consistent": "rnai_direction_consistent",
        }
    )
    .copy()
)

In [37]:
# =============================================================================
# Harmonize lineage-aware Entrez identifier type
# =============================================================================

crispr_lineage_for_integration["cross_platform_entrez_id"] = (
    pd.to_numeric(
        crispr_lineage_for_integration["cross_platform_entrez_id"],
        errors="raise",
    )
    .astype("Int64")
)

rnai_lineage_for_integration["cross_platform_entrez_id"] = (
    pd.to_numeric(
        rnai_lineage_for_integration["cross_platform_entrez_id"],
        errors="raise",
    )
    .astype("Int64")
)

In [38]:
# =============================================================================
# Annotate CRISPR lineage direction consistency
# =============================================================================

crispr_lineage_for_integration = (
    crispr_lineage_for_integration
    .merge(
        crispr_primary_for_integration[
            [
                "consensus_program_id",
                "cross_platform_entrez_id",
                "crispr_beta",
            ]
        ].rename(
            columns={
                "crispr_beta": "crispr_primary_beta",
            }
        ),
        on=[
            "consensus_program_id",
            "cross_platform_entrez_id",
        ],
        how="left",
        validate="many_to_one",
    )
)

crispr_lineage_for_integration[
    "crispr_direction_consistent"
] = (
    np.sign(
        crispr_lineage_for_integration[
            "crispr_lineage_beta"
        ]
    )
    == np.sign(
        crispr_lineage_for_integration[
            "crispr_primary_beta"
        ]
    )
)

### CRISPR lineage-direction consistency

Notebook 500 retained within-lineage CRISPR effect estimates but did not persist
an explicit direction-consistency indicator analogous to the one available in
the RNAi output.

For descriptive cross-platform lineage context, notebook 502 therefore derives
CRISPR direction consistency by comparing the sign of each frozen within-lineage
beta with the sign of the corresponding frozen primary CRISPR beta.

This operation does not refit any model, modify an effect estimate, introduce a
new threshold, or alter platform-specific significance or eligibility. The
derived indicator is used only to summarize lineage-direction patterns
descriptively.

In [39]:
# =============================================================================
# Summarize platform-specific lineage-aware context
# =============================================================================

crispr_lineage_summary = (
    crispr_lineage_for_integration
    .groupby(
        [
            "consensus_program_id",
            "cross_platform_entrez_id",
        ],
        observed=True,
    )
    .agg(
        crispr_evaluable_lineages=(
            "crispr_lineage",
            "nunique",
        ),
        crispr_direction_consistency_fraction=(
            "crispr_direction_consistent",
            "mean",
        ),
        crispr_lineage_beta_median=(
            "crispr_lineage_beta",
            "median",
        ),
        crispr_lineage_beta_q25=(
            "crispr_lineage_beta",
            lambda values: values.quantile(0.25),
        ),
        crispr_lineage_beta_q75=(
            "crispr_lineage_beta",
            lambda values: values.quantile(0.75),
        ),
        crispr_lineage_beta_min=(
            "crispr_lineage_beta",
            "min",
        ),
        crispr_lineage_beta_max=(
            "crispr_lineage_beta",
            "max",
        ),
    )
    .reset_index()
)

crispr_lineage_summary["crispr_lineage_beta_iqr"] = (
    crispr_lineage_summary["crispr_lineage_beta_q75"]
    - crispr_lineage_summary["crispr_lineage_beta_q25"]
)

rnai_lineage_summary = (
    rnai_lineage_for_integration
    .groupby(
        [
            "consensus_program_id",
            "cross_platform_entrez_id",
        ],
        observed=True,
    )
    .agg(
        rnai_evaluable_lineages=(
            "rnai_lineage",
            "nunique",
        ),
        rnai_direction_consistency_fraction=(
            "rnai_direction_consistent",
            "mean",
        ),
        rnai_lineage_beta_median=(
            "rnai_lineage_beta",
            "median",
        ),
        rnai_lineage_beta_q25=(
            "rnai_lineage_beta",
            lambda values: values.quantile(0.25),
        ),
        rnai_lineage_beta_q75=(
            "rnai_lineage_beta",
            lambda values: values.quantile(0.75),
        ),
        rnai_lineage_beta_min=(
            "rnai_lineage_beta",
            "min",
        ),
        rnai_lineage_beta_max=(
            "rnai_lineage_beta",
            "max",
        ),
    )
    .reset_index()
)

rnai_lineage_summary["rnai_lineage_beta_iqr"] = (
    rnai_lineage_summary["rnai_lineage_beta_q75"]
    - rnai_lineage_summary["rnai_lineage_beta_q25"]
)

In [40]:
# =============================================================================
# Integrate platform-specific lineage-aware context
# =============================================================================

integrated_primary_map = (
    integrated_primary_map
    .merge(
        crispr_lineage_summary,
        on=[
            "consensus_program_id",
            "cross_platform_entrez_id",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        rnai_lineage_summary,
        on=[
            "consensus_program_id",
            "cross_platform_entrez_id",
        ],
        how="left",
        validate="one_to_one",
    )
)

In [41]:
# =============================================================================
# Characterize lineage-aware evidence availability
# =============================================================================

lineage_context_availability = pd.Series(
    {
        "integrated_hypotheses": len(
            integrated_primary_map
        ),
        "crispr_lineage_context_available": (
            integrated_primary_map[
                "crispr_evaluable_lineages"
            ].notna().sum()
        ),
        "rnai_lineage_context_available": (
            integrated_primary_map[
                "rnai_evaluable_lineages"
            ].notna().sum()
        ),
        "both_lineage_contexts_available": (
            integrated_primary_map[
                "crispr_evaluable_lineages"
            ].notna()
            & integrated_primary_map[
                "rnai_evaluable_lineages"
            ].notna()
        ).sum(),
        "both_lineage_contexts_among_comparable": (
            integrated_primary_map[
                "crispr_evaluable_lineages"
            ].notna()
            & integrated_primary_map[
                "rnai_evaluable_lineages"
            ].notna()
            & integrated_primary_map[
                "comparability_status"
            ].eq("cross_platform_comparable")
        ).sum(),
    },
    name="value",
).to_frame()

lineage_context_availability

,value
integrated_hypotheses,54951
crispr_lineage_context_available,944
rnai_lineage_context_available,1686
both_lineage_contexts_available,118
both_lineage_contexts_among_comparable,118


In [42]:
# =============================================================================
# Inspect RNAi sensitivity integration fields
# =============================================================================

rnai_sensitivity_field_inventory = pd.DataFrame(
    {
        "source_adjusted_columns": pd.Series(
            rnai_source_adjusted.columns,
            dtype="string",
        ),
        "coverage90_columns": pd.Series(
            rnai_coverage90.columns,
            dtype="string",
        ),
    }
)

rnai_sensitivity_field_inventory

,source_adjusted_columns,coverage90_columns
0,consensus_program_id,consensus_program_id
1,gene_label,gene_label
2,source_adjusted_estimable,n_models
3,n_models,n_lineages
4,n_lineages,beta
5,n_source_patterns,standard_error
6,beta,ci_95_lower
7,standard_error,ci_95_upper
8,ci_95_lower,p_value
9,ci_95_upper,fdr_q_value


In [43]:
# =============================================================================
# Prepare RNAi sensitivity results for integration
# =============================================================================

rnai_source_adjusted_for_integration = (
    rnai_source_adjusted[
        [
            "consensus_program_id",
            "gene_label",
            "gene_symbol",
            "entrez_id",
            "source_adjusted_estimable",
            "beta",
            "fdr_q_value_source_adjusted",
            "fdr_significant_source_adjusted",
        ]
    ]
    .rename(
        columns={
            "gene_label": "rnai_gene_label",
            "gene_symbol": "rnai_gene_symbol",
            "entrez_id": "cross_platform_entrez_id",
            "beta": "rnai_source_adjusted_beta",
        }
    )
    .copy()
)

rnai_coverage90_for_integration = (
    rnai_coverage90[
        [
            "consensus_program_id",
            "gene_label",
            "gene_symbol",
            "entrez_id",
            "beta",
            "fdr_q_value_coverage90",
            "fdr_significant_coverage90",
        ]
    ]
    .rename(
        columns={
            "gene_label": "rnai_gene_label",
            "gene_symbol": "rnai_gene_symbol",
            "entrez_id": "cross_platform_entrez_id",
            "beta": "rnai_coverage90_beta",
        }
    )
    .copy()
)

In [44]:
# =============================================================================
# Harmonize RNAi sensitivity Entrez identifier type
# =============================================================================

rnai_source_adjusted_for_integration[
    "cross_platform_entrez_id"
] = (
    pd.to_numeric(
        rnai_source_adjusted_for_integration[
            "cross_platform_entrez_id"
        ],
        errors="raise",
    )
    .astype("Int64")
)

rnai_coverage90_for_integration[
    "cross_platform_entrez_id"
] = (
    pd.to_numeric(
        rnai_coverage90_for_integration[
            "cross_platform_entrez_id"
        ],
        errors="raise",
    )
    .astype("Int64")
)

In [45]:
# =============================================================================
# Integrate RNAi sensitivity annotations
# =============================================================================

integrated_primary_map = (
    integrated_primary_map
    .merge(
        rnai_source_adjusted_for_integration[
            [
                "consensus_program_id",
                "cross_platform_entrez_id",
                "source_adjusted_estimable",
                "rnai_source_adjusted_beta",
                "fdr_q_value_source_adjusted",
                "fdr_significant_source_adjusted",
            ]
        ],
        on=[
            "consensus_program_id",
            "cross_platform_entrez_id",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        rnai_coverage90_for_integration[
            [
                "consensus_program_id",
                "cross_platform_entrez_id",
                "rnai_coverage90_beta",
                "fdr_q_value_coverage90",
                "fdr_significant_coverage90",
            ]
        ],
        on=[
            "consensus_program_id",
            "cross_platform_entrez_id",
        ],
        how="left",
        validate="one_to_one",
    )
)

In [46]:
# =============================================================================
# Characterize RNAi sensitivity availability
# =============================================================================

rnai_primary_available = (
    integrated_primary_map["rnai_beta"].notna()
)

rnai_source_adjusted_available = (
    integrated_primary_map[
        "source_adjusted_estimable"
    ].eq(True)
)

rnai_coverage90_available = (
    integrated_primary_map[
        "rnai_coverage90_beta"
    ].notna()
)

rnai_sensitivity_availability = pd.Series(
    {
        "rnai_primary_hypotheses": (
            rnai_primary_available.sum()
        ),
        "source_adjusted_estimable": (
            rnai_source_adjusted_available.sum()
        ),
        "coverage90_hypotheses": (
            rnai_coverage90_available.sum()
        ),
        "source_adjusted_fraction_primary": (
            rnai_source_adjusted_available.sum()
            / rnai_primary_available.sum()
        ),
        "coverage90_fraction_primary": (
            rnai_coverage90_available.sum()
            / rnai_primary_available.sum()
        ),
    },
    name="value",
).to_frame()

rnai_sensitivity_availability

,value
rnai_primary_hypotheses,37794.000000
source_adjusted_estimable,37794.000000
coverage90_hypotheses,18279.000000
source_adjusted_fraction_primary,1.000000
coverage90_fraction_primary,0.483648


In [47]:
# =============================================================================
# Compare RNAi primary and sensitivity significance
# =============================================================================

rnai_primary_significant = (
    integrated_primary_map[
        "rnai_fdr_significant"
    ].eq(True)
)

rnai_source_significant = (
    integrated_primary_map[
        "fdr_significant_source_adjusted"
    ].eq(True)
)

rnai_coverage90_significant = (
    integrated_primary_map[
        "fdr_significant_coverage90"
    ].eq(True)
)

rnai_sensitivity_significance_summary = pd.DataFrame(
    [
        {
            "sensitivity": "source_adjusted",
            "evaluable_hypotheses": rnai_source_adjusted_available.sum(),
            "both_significant": (
                rnai_source_adjusted_available
                & rnai_primary_significant
                & rnai_source_significant
            ).sum(),
            "primary_only": (
                rnai_source_adjusted_available
                & rnai_primary_significant
                & rnai_source_significant.eq(False)
            ).sum(),
            "sensitivity_only": (
                rnai_source_adjusted_available
                & rnai_primary_significant.eq(False)
                & rnai_source_significant
            ).sum(),
            "neither_significant": (
                rnai_source_adjusted_available
                & rnai_primary_significant.eq(False)
                & rnai_source_significant.eq(False)
            ).sum(),
        },
        {
            "sensitivity": "coverage90",
            "evaluable_hypotheses": rnai_coverage90_available.sum(),
            "both_significant": (
                rnai_coverage90_available
                & rnai_primary_significant
                & rnai_coverage90_significant
            ).sum(),
            "primary_only": (
                rnai_coverage90_available
                & rnai_primary_significant
                & rnai_coverage90_significant.eq(False)
            ).sum(),
            "sensitivity_only": (
                rnai_coverage90_available
                & rnai_primary_significant.eq(False)
                & rnai_coverage90_significant
            ).sum(),
            "neither_significant": (
                rnai_coverage90_available
                & rnai_primary_significant.eq(False)
                & rnai_coverage90_significant.eq(False)
            ).sum(),
        },
    ]
)

rnai_sensitivity_significance_summary

,sensitivity,evaluable_hypotheses,both_significant,primary_only,sensitivity_only,neither_significant
0,source_adjusted,37794,1414,272,93,36015
1,coverage90,18279,902,0,69,17308


In [48]:
# =============================================================================
# Standardize RNAi sensitivity significance flags
# =============================================================================

integrated_primary_map[
    "rnai_source_adjusted_fdr_significant"
] = (
    integrated_primary_map[
        "fdr_significant_source_adjusted"
    ]
    .eq(True)
    .where(rnai_source_adjusted_available)
    .astype("boolean")
)

integrated_primary_map[
    "rnai_coverage90_fdr_significant"
] = (
    integrated_primary_map[
        "fdr_significant_coverage90"
    ]
    .eq(True)
    .where(rnai_coverage90_available)
    .astype("boolean")
)

In [49]:
# =============================================================================
# Characterize RNAi sensitivity support among concordant putative vulnerabilities
# =============================================================================

concordant_putative_mask = (
    integrated_primary_map[
        "cross_platform_evidence_category"
    ].eq("concordant_putative_vulnerability")
)

concordant_putative_sensitivity = (
    integrated_primary_map.loc[
        concordant_putative_mask
    ]
    .copy()
)

concordant_putative_sensitivity_summary = pd.Series(
    {
        "concordant_putative_associations": len(
            concordant_putative_sensitivity
        ),
        "source_adjusted_evaluable": (
            concordant_putative_sensitivity[
                "source_adjusted_estimable"
            ].eq(True).sum()
        ),
        "source_adjusted_fdr_significant": (
            concordant_putative_sensitivity[
                "rnai_source_adjusted_fdr_significant"
            ].eq(True).sum()
        ),
        "coverage90_evaluable": (
            concordant_putative_sensitivity[
                "rnai_coverage90_beta"
            ].notna().sum()
        ),
        "coverage90_fdr_significant": (
            concordant_putative_sensitivity[
                "rnai_coverage90_fdr_significant"
            ].eq(True).sum()
        ),
    },
    name="value",
).to_frame()

concordant_putative_sensitivity_summary

,value
concordant_putative_associations,35
source_adjusted_evaluable,35
source_adjusted_fdr_significant,27
coverage90_evaluable,29
coverage90_fdr_significant,29


In [50]:
# =============================================================================
# Summarize RNAi sensitivity context by consensus program
# =============================================================================

concordant_putative_sensitivity_by_program = (
    concordant_putative_sensitivity
    .groupby(
        "consensus_program_id",
        observed=True,
    )
    .agg(
        concordant_putative_associations=(
            "cross_platform_entrez_id",
            "size",
        ),
        source_adjusted_evaluable=(
            "source_adjusted_estimable",
            lambda values: values.eq(True).sum(),
        ),
        source_adjusted_fdr_significant=(
            "rnai_source_adjusted_fdr_significant",
            lambda values: values.eq(True).sum(),
        ),
        coverage90_evaluable=(
            "rnai_coverage90_beta",
            lambda values: values.notna().sum(),
        ),
        coverage90_fdr_significant=(
            "rnai_coverage90_fdr_significant",
            lambda values: values.eq(True).sum(),
        ),
    )
    .reset_index()
)

concordant_putative_sensitivity_by_program

,consensus_program_id,concordant_putative_associations,source_adjusted_evaluable,source_adjusted_fdr_significant,coverage90_evaluable,coverage90_fdr_significant
0,CONSENSUS_TX_01,16,16,11,12,12
1,CONSENSUS_TX_02,6,6,5,5,5
2,CONSENSUS_TX_03,13,13,11,12,12


In [51]:
# =============================================================================
# Characterize lineage context among concordant putative vulnerabilities
# =============================================================================

concordant_putative_lineage_summary = pd.DataFrame(
    [
        {
            "platform": "CRISPR",
            "associations_with_lineage_context": (
                concordant_putative_sensitivity[
                    "crispr_evaluable_lineages"
                ].notna().sum()
            ),
            "evaluable_lineages_mean": (
                concordant_putative_sensitivity[
                    "crispr_evaluable_lineages"
                ].mean()
            ),
            "evaluable_lineages_median": (
                concordant_putative_sensitivity[
                    "crispr_evaluable_lineages"
                ].median()
            ),
            "direction_consistency_mean": (
                concordant_putative_sensitivity[
                    "crispr_direction_consistency_fraction"
                ].mean()
            ),
            "direction_consistency_median": (
                concordant_putative_sensitivity[
                    "crispr_direction_consistency_fraction"
                ].median()
            ),
        },
        {
            "platform": "RNAi",
            "associations_with_lineage_context": (
                concordant_putative_sensitivity[
                    "rnai_evaluable_lineages"
                ].notna().sum()
            ),
            "evaluable_lineages_mean": (
                concordant_putative_sensitivity[
                    "rnai_evaluable_lineages"
                ].mean()
            ),
            "evaluable_lineages_median": (
                concordant_putative_sensitivity[
                    "rnai_evaluable_lineages"
                ].median()
            ),
            "direction_consistency_mean": (
                concordant_putative_sensitivity[
                    "rnai_direction_consistency_fraction"
                ].mean()
            ),
            "direction_consistency_median": (
                concordant_putative_sensitivity[
                    "rnai_direction_consistency_fraction"
                ].median()
            ),
        },
    ]
)

concordant_putative_lineage_summary

,platform,associations_with_lineage_context,evaluable_lineages_mean,evaluable_lineages_median,direction_consistency_mean,direction_consistency_median
0,CRISPR,35,13.000000,13.0,0.747253,0.769231
1,RNAi,35,10.657143,11.0,0.756421,0.777778


In [52]:
# =============================================================================
# Summarize lineage context by consensus program
# =============================================================================

concordant_putative_lineage_by_program = (
    concordant_putative_sensitivity
    .groupby(
        "consensus_program_id",
        observed=True,
    )
    .agg(
        n_associations=(
            "cross_platform_entrez_id",
            "size",
        ),
        crispr_evaluable_lineages_mean=(
            "crispr_evaluable_lineages",
            "mean",
        ),
        crispr_direction_consistency_mean=(
            "crispr_direction_consistency_fraction",
            "mean",
        ),
        rnai_evaluable_lineages_mean=(
            "rnai_evaluable_lineages",
            "mean",
        ),
        rnai_direction_consistency_mean=(
            "rnai_direction_consistency_fraction",
            "mean",
        ),
    )
    .reset_index()
)

concordant_putative_lineage_by_program

,consensus_program_id,n_associations,crispr_evaluable_lineages_mean,crispr_direction_consistency_mean,rnai_evaluable_lineages_mean,rnai_direction_consistency_mean
0,CONSENSUS_TX_01,16,13.0,0.730769,10.500000,0.779040
1,CONSENSUS_TX_02,6,13.0,0.756410,10.666667,0.781145
2,CONSENSUS_TX_03,13,13.0,0.763314,10.846154,0.717172


In [53]:
# =============================================================================
# Construct final integrated vulnerability map
# =============================================================================

integrated_primary_map["cross_platform_gene_symbol"] = (
    integrated_primary_map["crispr_gene_symbol"]
    .where(
        integrated_primary_map[
            "gene_symbol_exact_match"
        ].eq(True)
    )
    .astype("string")
)

integrated_vulnerability_map = (
    integrated_primary_map[
        [
            "consensus_program_id",
            "cross_platform_entrez_id",
            "cross_platform_gene_symbol",
            "comparability_status",
            "hypothesis_membership",
            "gene_symbol_exact_match",
            "cross_platform_evidence_category",
            "cross_platform_direction_concordant",
            "both_platforms_fdr_significant",
            "crispr_gene_label",
            "crispr_gene_symbol",
            "crispr_n_models",
            "crispr_beta",
            "crispr_standard_error",
            "crispr_p_value",
            "crispr_fdr_q_value",
            "crispr_fdr_significant",
            "crispr_effect_direction",
            "rnai_gene_label",
            "rnai_gene_symbol",
            "rnai_n_models",
            "rnai_n_lineages",
            "rnai_observed_models",
            "rnai_coverage_fraction",
            "rnai_beta",
            "rnai_standard_error",
            "rnai_ci_95_lower",
            "rnai_ci_95_upper",
            "rnai_p_value",
            "rnai_fdr_q_value",
            "rnai_fdr_significant",
            "rnai_effect_direction",
            "crispr_evaluable_lineages",
            "crispr_direction_consistency_fraction",
            "crispr_lineage_beta_median",
            "crispr_lineage_beta_iqr",
            "crispr_lineage_beta_min",
            "crispr_lineage_beta_max",
            "rnai_evaluable_lineages",
            "rnai_direction_consistency_fraction",
            "rnai_lineage_beta_median",
            "rnai_lineage_beta_iqr",
            "rnai_lineage_beta_min",
            "rnai_lineage_beta_max",
            "source_adjusted_estimable",
            "rnai_source_adjusted_beta",
            "fdr_q_value_source_adjusted",
            "rnai_source_adjusted_fdr_significant",
            "rnai_coverage90_beta",
            "fdr_q_value_coverage90",
            "rnai_coverage90_fdr_significant",
        ]
    ]
    .sort_values(
        [
            "consensus_program_id",
            "cross_platform_entrez_id",
        ]
    )
    .reset_index(drop=True)
)

integrated_vulnerability_map.shape

(54951, 51)

In [54]:
# =============================================================================
# Prepare concordant putative-vulnerability handoff
# =============================================================================

concordant_putative_vulnerability_associations = (
    integrated_vulnerability_map.loc[
        integrated_vulnerability_map[
            "cross_platform_evidence_category"
        ].eq("concordant_putative_vulnerability")
    ]
    .copy()
    .sort_values(
        [
            "consensus_program_id",
            "cross_platform_entrez_id",
        ]
    )
    .reset_index(drop=True)
)

concordant_putative_vulnerability_associations.shape

(35, 51)

In [55]:
# =============================================================================
# Verify final integrated-map integrity
# =============================================================================

final_map_integrity = pd.Series(
    {
        "rows": len(integrated_vulnerability_map),
        "columns": integrated_vulnerability_map.shape[1],
        "missing_key_rows": (
            integrated_vulnerability_map[
                [
                    "consensus_program_id",
                    "cross_platform_entrez_id",
                ]
            ]
            .isna()
            .any(axis=1)
            .sum()
        ),
        "duplicated_key_rows": (
            integrated_vulnerability_map
            .duplicated(
                [
                    "consensus_program_id",
                    "cross_platform_entrez_id",
                ],
                keep=False,
            )
            .sum()
        ),
        "classified_rows": (
            integrated_vulnerability_map[
                "cross_platform_evidence_category"
            ]
            .notna()
            .sum()
        ),
    },
    name="value",
).to_frame()

final_map_integrity

,value
rows,54951
columns,51
missing_key_rows,0
duplicated_key_rows,0
classified_rows,54951


In [56]:
# =============================================================================
# Output paths
# =============================================================================

CROSS_PLATFORM_GENE_MAP_PATH = (
    Paths.dependencies
    / "502_cross_platform_gene_map.csv"
)

INTEGRATED_VULNERABILITY_MAP_PATH = (
    Paths.functional_vulnerabilities
    / "502_integrated_vulnerability_map.parquet"
)

CROSS_PLATFORM_EVIDENCE_SUMMARY_PATH = (
    Paths.functional_vulnerabilities
    / "502_cross_platform_evidence_summary.csv"
)

CONCORDANT_PUTATIVE_VULNERABILITIES_PATH = (
    Paths.functional_vulnerabilities
    / "502_concordant_putative_vulnerability_associations.csv"
)

INTEGRATED_VULNERABILITY_METADATA_PATH = (
    Paths.functional_vulnerabilities
    / "502_integrated_vulnerability_metadata.json"
)

In [57]:
# =============================================================================
# Prepare cross-platform evidence summary artifact
# =============================================================================

program_category_index = pd.MultiIndex.from_product(
    [
        FROZEN_CONSENSUS_PROGRAM_IDS,
        CROSS_PLATFORM_EVIDENCE_CATEGORIES,
    ],
    names=[
        "consensus_program_id",
        "cross_platform_evidence_category",
    ],
)

cross_platform_evidence_by_program = (
    cross_platform_evidence_summary
    .set_index(
        [
            "consensus_program_id",
            "cross_platform_evidence_category",
        ]
    )
    .reindex(
        program_category_index,
        fill_value=0,
    )
    .reset_index()
)

cross_platform_evidence_by_program.insert(
    0,
    "scope",
    "program",
)

cross_platform_evidence_global = (
    cross_platform_evidence_global_summary
    .assign(
        scope="global",
        consensus_program_id=pd.NA,
    )
    [
        [
            "scope",
            "consensus_program_id",
            "cross_platform_evidence_category",
            "n_hypotheses",
        ]
    ]
)

cross_platform_evidence_summary_artifact = pd.concat(
    [
        cross_platform_evidence_global,
        cross_platform_evidence_by_program,
    ],
    ignore_index=True,
)

cross_platform_evidence_summary_artifact

,scope,consensus_program_id,cross_platform_evidence_category,n_hypotheses
0,global,<NA>,concordant_putative_vulnerability,35
1,global,<NA>,concordant_weaker_dependency,77
2,global,<NA>,crispr_supported_only,532
3,global,<NA>,rnai_supported_only,1452
4,global,<NA>,directionally_discordant,6
5,global,<NA>,non_significant_both,32356
6,global,<NA>,not_cross_platform_comparable,20493
7,program,CONSENSUS_TX_01,concordant_putative_vulnerability,16
8,program,CONSENSUS_TX_01,concordant_weaker_dependency,12
9,program,CONSENSUS_TX_01,crispr_supported_only,164


In [58]:
# =============================================================================
# Prepare integrated vulnerability metadata
# =============================================================================

crispr_model_ids = set(
    crispr_model_cohort["ModelID"].astype("string")
)

rnai_model_ids = set(
    rnai_model_cohort["ModelID"].astype("string")
)

shared_model_ids = crispr_model_ids & rnai_model_ids

comparable_integrated_map = integrated_vulnerability_map.loc[
    integrated_vulnerability_map[
        "comparability_status"
    ].eq("cross_platform_comparable")
].copy()

global_evidence_counts = (
    cross_platform_evidence_summary_artifact.loc[
        cross_platform_evidence_summary_artifact[
            "scope"
        ].eq("global")
    ]
    .set_index("cross_platform_evidence_category")[
        "n_hypotheses"
    ]
    .astype(int)
    .to_dict()
)

program_effect_correspondence = {
    program_id: float(
        program_frame["crispr_beta"].corr(
            program_frame["rnai_beta"],
            method="spearman",
        )
    )
    for program_id, program_frame in (
        comparable_integrated_map.groupby(
            "consensus_program_id",
            sort=False,
        )
    )
}

bilateral_significant = comparable_integrated_map[
    "both_platforms_fdr_significant"
].eq(True)

integrated_vulnerability_metadata = {
    "notebook": "502_integrated_vulnerability_mapping",
    "analysis_scope": (
        "Cross-platform integration of frozen CRISPR and RNAi "
        "program-dependency association layers."
    ),
    "consensus_program_ids": FROZEN_CONSENSUS_PROGRAM_IDS,
    "integration_unit": (
        "frozen consensus_program_id × Entrez gene identifier"
    ),
    "input_artifacts": [
        str(project_relative_path(path))
        for path in [
            CRISPR_MODEL_COHORT_PATH,
            CRISPR_GENE_COVERAGE_PATH,
            CRISPR_PRIMARY_ASSOCIATIONS_PATH,
            CRISPR_WITHIN_LINEAGE_PATH,
            RNAI_MODEL_COHORT_PATH,
            RNAI_GENE_ELIGIBILITY_PATH,
            RNAI_PRIMARY_ASSOCIATIONS_PATH,
            RNAI_WITHIN_LINEAGE_PATH,
            RNAI_SOURCE_ADJUSTED_PATH,
            RNAI_COVERAGE90_PATH,
        ]
    ],
    "output_artifacts": [
        str(project_relative_path(path))
        for path in [
            CROSS_PLATFORM_GENE_MAP_PATH,
            INTEGRATED_VULNERABILITY_MAP_PATH,
            CROSS_PLATFORM_EVIDENCE_SUMMARY_PATH,
            CONCORDANT_PUTATIVE_VULNERABILITIES_PATH,
            INTEGRATED_VULNERABILITY_METADATA_PATH,
        ]
    ],
    "platform_specific_eligibility": {
        "crispr_primary_coverage_threshold": 0.90,
        "rnai_primary_coverage_threshold": 0.75,
        "rnai_primary_target_scope": "single-gene DEMETER2 targets",
        "common_cross_platform_coverage_threshold_applied": False,
        "rnai_coverage90_role": (
            "Prespecified sensitivity analysis only; does not "
            "redefine primary eligibility or cross-platform comparability."
        ),
    },
    "gene_mapping": {
        "primary_identifier": "Entrez ID",
        "symbol_used_for_mapping": False,
        "cross_platform_gene_symbol_rule": (
            "Populated only when CRISPR and RNAi gene symbols "
            "match exactly; Entrez ID defines correspondence."
        ),
        "crispr_eligible_genes": int(
            cross_platform_gene_map[
                "comparability_status"
            ]
            .isin(
                [
                    "cross_platform_comparable",
                    "crispr_only",
                ]
            )
            .sum()
        ),
        "rnai_eligible_genes": int(
            cross_platform_gene_map[
                "comparability_status"
            ]
            .isin(
                [
                    "cross_platform_comparable",
                    "rnai_only",
                ]
            )
            .sum()
        ),
        "shared_eligible_genes": int(
            cross_platform_gene_map[
                "comparability_status"
            ]
            .eq("cross_platform_comparable")
            .sum()
        ),
        "crispr_only_genes": int(
            cross_platform_gene_map[
                "comparability_status"
            ]
            .eq("crispr_only")
            .sum()
        ),
        "rnai_only_genes": int(
            cross_platform_gene_map[
                "comparability_status"
            ]
            .eq("rnai_only")
            .sum()
        ),
        "shared_gene_symbol_mismatches": int(
            cross_platform_gene_map.loc[
                cross_platform_gene_map[
                    "comparability_status"
                ].eq("cross_platform_comparable"),
                "gene_symbol_exact_match",
            ]
            .eq(False)
            .sum()
        ),
    },
    "model_overlap": {
        "crispr_models": len(crispr_model_ids),
        "rnai_models": len(rnai_model_ids),
        "shared_models": len(shared_model_ids),
        "shared_fraction_crispr": (
            len(shared_model_ids) / len(crispr_model_ids)
        ),
        "shared_fraction_rnai": (
            len(shared_model_ids) / len(rnai_model_ids)
        ),
    },
    "hypothesis_universe": {
        "integrated_hypotheses": len(
            integrated_vulnerability_map
        ),
        "cross_platform_comparable": int(
            integrated_vulnerability_map[
                "comparability_status"
            ]
            .eq("cross_platform_comparable")
            .sum()
        ),
        "crispr_only": int(
            integrated_vulnerability_map[
                "comparability_status"
            ]
            .eq("crispr_only")
            .sum()
        ),
        "rnai_only": int(
            integrated_vulnerability_map[
                "comparability_status"
            ]
            .eq("rnai_only")
            .sum()
        ),
    },
    "cross_platform_evidence_counts": global_evidence_counts,
    "effect_size_correspondence": {
        "global_spearman_beta": float(
            comparable_integrated_map[
                "crispr_beta"
            ].corr(
                comparable_integrated_map[
                    "rnai_beta"
                ],
                method="spearman",
            )
        ),
        "program_specific_spearman_beta": (
            program_effect_correspondence
        ),
    },
    "bilateral_significant_directionality": {
        "both_platforms_fdr_significant": int(
            bilateral_significant.sum()
        ),
        "direction_concordant": int(
            comparable_integrated_map.loc[
                bilateral_significant,
                "cross_platform_direction_concordant",
            ]
            .eq(True)
            .sum()
        ),
        "direction_discordant": int(
            comparable_integrated_map.loc[
                bilateral_significant,
                "cross_platform_direction_concordant",
            ]
            .eq(False)
            .sum()
        ),
        "concordance_fraction": float(
            comparable_integrated_map.loc[
                bilateral_significant,
                "cross_platform_direction_concordant",
            ].mean()
        ),
    },
    "concordant_putative_vulnerability_context": {
        "n_associations": len(
            concordant_putative_vulnerability_associations
        ),
        "n_unique_genes": int(
            concordant_putative_vulnerability_associations[
                "cross_platform_entrez_id"
            ].nunique()
        ),
        "rnai_source_adjusted_fdr_significant": int(
            concordant_putative_vulnerability_associations[
                "rnai_source_adjusted_fdr_significant"
            ]
            .eq(True)
            .sum()
        ),
        "rnai_coverage90_evaluable": int(
            concordant_putative_vulnerability_associations[
                "rnai_coverage90_beta"
            ]
            .notna()
            .sum()
        ),
        "rnai_coverage90_fdr_significant": int(
            concordant_putative_vulnerability_associations[
                "rnai_coverage90_fdr_significant"
            ]
            .eq(True)
            .sum()
        ),
    },
    "integration_constraints": {
        "dependency_score_pooling": False,
        "coefficient_pooling": False,
        "p_value_combination": False,
        "q_value_combination": False,
        "joint_cross_platform_fdr": False,
        "composite_vulnerability_score": False,
        "cross_platform_ranking": False,
        "concordance_used_as_significance_gate": False,
        "lineage_context_used_as_filter": False,
        "rnai_sensitivities_used_as_reclassification_gate": False,
    },
    "interpretation": {
        "cross_platform_concordance": (
            "Complementary computational evidence, not "
            "independent validation."
        ),
        "single_platform_findings": (
            "Retained as valid platform-specific computational "
            "associations."
        ),
        "directional_discordance": (
            "Retained explicitly and not resolved by post-hoc "
            "platform preference."
        ),
        "lineage_context": (
            "Descriptive and result-conditioned; no new "
            "consistency threshold applied."
        ),
    },
}

In [59]:
# =============================================================================
# Persist integrated vulnerability artifacts
# =============================================================================

cross_platform_gene_map.to_csv(
    CROSS_PLATFORM_GENE_MAP_PATH,
    index=False,
)

integrated_vulnerability_map.to_parquet(
    INTEGRATED_VULNERABILITY_MAP_PATH,
    index=False,
)

cross_platform_evidence_summary_artifact.to_csv(
    CROSS_PLATFORM_EVIDENCE_SUMMARY_PATH,
    index=False,
)

concordant_putative_vulnerability_associations.to_csv(
    CONCORDANT_PUTATIVE_VULNERABILITIES_PATH,
    index=False,
)

with INTEGRATED_VULNERABILITY_METADATA_PATH.open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        integrated_vulnerability_metadata,
        handle,
        indent=2,
        sort_keys=True,
    )

In [60]:
# =============================================================================
# Verify persisted integrated vulnerability artifacts
# =============================================================================

published_artifact_paths = [
    CROSS_PLATFORM_GENE_MAP_PATH,
    INTEGRATED_VULNERABILITY_MAP_PATH,
    CROSS_PLATFORM_EVIDENCE_SUMMARY_PATH,
    CONCORDANT_PUTATIVE_VULNERABILITIES_PATH,
    INTEGRATED_VULNERABILITY_METADATA_PATH,
]

publication_check = pd.DataFrame(
    {
        "artifact": [
            project_relative_path(path)
            for path in published_artifact_paths
        ],
        "exists": [
            path.exists()
            for path in published_artifact_paths
        ],
        "size_mib": [
            path.stat().st_size / (1024 ** 2)
            if path.exists()
            else np.nan
            for path in published_artifact_paths
        ],
    }
)

reloaded_integrated_map = pd.read_parquet(
    INTEGRATED_VULNERABILITY_MAP_PATH
)

publication_integrity = pd.Series(
    {
        "reloaded_rows": len(reloaded_integrated_map),
        "reloaded_columns": reloaded_integrated_map.shape[1],
        "missing_key_rows": (
            reloaded_integrated_map[
                [
                    "consensus_program_id",
                    "cross_platform_entrez_id",
                ]
            ]
            .isna()
            .any(axis=1)
            .sum()
        ),
        "duplicated_key_rows": (
            reloaded_integrated_map
            .duplicated(
                [
                    "consensus_program_id",
                    "cross_platform_entrez_id",
                ],
                keep=False,
            )
            .sum()
        ),
    },
    name="value",
).to_frame()

display(publication_check)
display(publication_integrity)

,artifact,exists,size_mib
0,data/interim/dependencies/502_cross_platform_g...,True,1.353237
1,data/processed/functional_vulnerabilities/502_...,True,6.051768
2,data/processed/functional_vulnerabilities/502_...,True,0.001426
3,data/processed/functional_vulnerabilities/502_...,True,0.025942
4,data/processed/functional_vulnerabilities/502_...,True,0.004613


,value
reloaded_rows,54951
reloaded_columns,51
missing_key_rows,0
duplicated_key_rows,0


## Integrated vulnerability mapping — summary

Notebook 502 integrated the frozen CRISPR and RNAi association layers without
refitting, pooling, or retrospectively redefining either platform-specific
analysis.

The final cross-platform map contains 54,951 program × gene analytical units
across the three frozen consensus programs.

### Cross-platform comparability

The primary CRISPR and RNAi eligible gene universes contained 17,205 and 12,598
genes, respectively. Exact Entrez ID matching identified 11,486 genes eligible
in both platforms.

This produced:

- 34,458 cross-platform-comparable program × gene hypotheses;
- 17,157 CRISPR-only hypotheses;
- 3,336 RNAi-only hypotheses.

Gene mapping was deterministic and based exclusively on exact Entrez IDs.
Among shared genes, 207 Entrez-matched genes had different gene symbols across
the two upstream annotation layers. These symbol differences were retained
explicitly and were not resolved post hoc.

### Cross-platform evidence

Among the 34,458 hypotheses evaluable in both platforms:

- 35 showed concordant FDR-significant associations oriented toward stronger
  dependency at higher consensus-program score;
- 77 showed concordant FDR-significant associations oriented toward weaker
  dependency;
- 6 were FDR-significant in both platforms but directionally discordant;
- 532 were FDR-significant in CRISPR only;
- 1,452 were FDR-significant in RNAi only;
- 32,356 were not FDR-significant in either platform.

Thus, 118 associations were FDR-significant in both CRISPR and RNAi, of which
112 (94.9%) had concordant effect direction.

This high directional agreement is conditional on bilateral significance and
must not be interpreted as an independent validation rate.

Across the complete cross-platform-comparable universe, directional agreement
was substantially weaker and the continuous correspondence between effect sizes
was modest (global Spearman rho = 0.105). Program-specific Spearman correlations
were similarly modest, ranging from approximately 0.097 to 0.140.

Consistent with the prespecified non-pooling design, the modest continuous
effect-size correspondence indicates that CRISPR and RNAi effect estimates
should remain separate platform-specific evidence dimensions rather than be
treated as interchangeable measurements or combined into a pooled coefficient,
composite score, or cross-platform ranking.

### Concordant putative-vulnerability associations

Thirty-five program × gene associations showed FDR-significant stronger-
dependency-oriented effects in both platforms, corresponding to 34 unique genes.

Only PRKAR1A recurred across more than one consensus program, occurring with
CONSENSUS_TX_02 and CONSENSUS_TX_03. Recurrence across programs is descriptive
and is not treated as an independent validation criterion.

RNAi sensitivity analyses provided additional platform-specific context:

- all 35 concordant associations were evaluable in the source-adjusted model,
  with 27 remaining FDR-significant;
- 29 were eligible under the >=90% RNAi coverage sensitivity, and all 29
  remained FDR-significant.

Associations not evaluable under the stricter coverage sensitivity were not
classified as unsupported.

### Lineage-aware context

All 35 concordant putative-vulnerability associations had lineage-aware
characterization available in both platforms.

Mean directional consistency across evaluable lineages was approximately 0.75
in both CRISPR and RNAi, with program-specific averages remaining below complete
uniformity.

Accordingly, these associations should be interpreted as potentially
context-dependent rather than as uniform pan-cancer dependencies. No new
lineage-consistency or heterogeneity threshold was introduced in this notebook.

### Interpretation boundary

Cross-platform concordance provides complementary computational evidence across
two distinct functional-genomics assays, but it does not constitute independent
validation.

CRISPR and RNAi share substantial biological context and partially overlapping
cell-line populations: 367 models were present in both cohorts, corresponding
to 68.1% of the CRISPR cohort and 82.8% of the RNAi cohort.

Single-platform and directionally discordant findings therefore remain valid
components of the integrated evidence map and were not discarded or resolved
through post-hoc platform preference.

The outputs of notebook 502 constitute a frozen, platform-aware functional-
vulnerability evidence layer for downstream analyses. They do not represent
validated targets, causal dependencies, therapeutic-response predictors, or a
combined vulnerability ranking.